# TP4 — Classification d'émotions avec RoBERTa-base

**Objectif :** classifier des tweets selon 6 émotions (`sadness`, `joy`, `love`, `anger`, `fear`, `surprise`) à partir du dataset `emotion` de Hugging Face.

**Approche différente :**
- on utilise **RoBERTa-base** (au lieu du classique DistilBERT),
- on **gère explicitement le déséquilibre des classes** via un `WeightedRandomSampler` et le suivi du **F1 macro**.


## 1. Imports et configuration

On importe `transformers`, `datasets`, `torch`, `sklearn` et on fixe la graine (`seed=42`).

In [ ]:
# Bibliothèques de base
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

# Hugging Face
from datasets import load_dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# Métriques scikit-learn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# On fixe la graine pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

## 2. Chargement du dataset

Le dataset `emotion` fournit directement les splits train / validation / test avec un label entier (0 à 5).

In [ ]:
# Chargement du dataset emotion depuis Hugging Face
dataset = load_dataset("emotion")

# Récupération des noms de classes dans l'ordre des labels
CLASS_NAMES = dataset["train"].features["label"].names
NUM_LABELS = len(CLASS_NAMES)
print(f"Classes ({NUM_LABELS}) : {CLASS_NAMES}")
print(dataset)
dataset["train"][0]

## 3. Analyse du déséquilibre des classes

On visualise la distribution des classes dans le train et on calcule des **poids inversement proportionnels** à leur fréquence, qui serviront ensuite à pondérer l'échantillonnage.

In [ ]:
# Comptage des occurrences de chaque classe dans le train
train_labels = np.array(dataset["train"]["label"])
counts = np.bincount(train_labels, minlength=NUM_LABELS)

# Barplot de la distribution des classes
plt.figure(figsize=(9, 5))
sns.barplot(x=CLASS_NAMES, y=counts)
plt.title("Distribution des classes (train)")
plt.ylabel("Nombre d'exemples")
plt.tight_layout()
plt.show()

for name, c in zip(CLASS_NAMES, counts):
    print(f"{name:<10} : {c}")

In [ ]:
# Poids par classe = inverse de la fréquence (les classes rares pèsent davantage)
class_weights = 1.0 / counts
# Poids attribué à chaque exemple du train selon sa classe
sample_weights = class_weights[train_labels]
print("Poids par classe :", np.round(class_weights / class_weights.sum(), 4))

## 4. Tokenisation

On tokenise les textes avec `RobertaTokenizer` : troncature et padding à `max_length=128`.

In [ ]:
# Chargement du tokenizer RoBERTa-base
MODEL_NAME = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    """Tokenise un batch de textes (troncature + padding à 128 tokens)."""
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

# Application de la tokenisation sur tous les splits
tokenized = dataset.map(tokenize_fn, batched=True)

# On garde uniquement les colonnes utiles au modèle et on fixe le format torch
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## 5. Modèle

`RobertaForSequenceClassification` avec une tête de classification à 6 labels (poids de la tête initialisés aléatoirement).

In [ ]:
# Chargement de RoBERTa-base avec une tête de classification à 6 classes
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: name for i, name in enumerate(CLASS_NAMES)},
    label2id={name: i for i, name in enumerate(CLASS_NAMES)},
)

## 6. Entraînement avec le Trainer Hugging Face

On configure :
- `TrainingArguments` : 3 époques, batch=16, lr=2e-5, warmup_ratio=0.1, évaluation/sauvegarde par époque ;
- un **`WeightedRandomSampler`** pour rééquilibrer l'échantillonnage des classes (via un Trainer personnalisé) ;
- `EarlyStoppingCallback` (patience=2) sur le F1 macro de validation ;
- une fonction de métriques : accuracy + F1 macro + F1 par classe.

In [ ]:
def compute_metrics(eval_pred):
    """Calcule accuracy, F1 macro et F1 par classe à partir des prédictions du Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),  # surveillé par l'early stopping
    }
    # F1 détaillé par classe (utile pour le suivi des classes rares)
    f1_per = f1_score(labels, preds, average=None, labels=list(range(NUM_LABELS)))
    for name, val in zip(CLASS_NAMES, f1_per):
        metrics[f"f1_{name}"] = val
    return metrics

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer personnalisé qui remplace l'échantillonnage du train par un WeightedRandomSampler."""
    def __init__(self, *args, sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._sample_weights = sample_weights

    def get_train_dataloader(self):
        # Échantillonnage pondéré : les classes rares sont tirées plus souvent
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(self._sample_weights, dtype=torch.double),
            num_samples=len(self._sample_weights),
            replacement=True,
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.per_device_train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
        )

In [ ]:
# Configuration de l'entraînement
training_args = TrainingArguments(
    output_dir="./roberta_emotions",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,                 # montée progressive du lr sur 10% des steps
    eval_strategy="epoch",            # évaluation à chaque époque
    save_strategy="epoch",            # sauvegarde à chaque époque
    load_best_model_at_end=True,      # recharge le meilleur modèle à la fin
    metric_for_best_model="f1",       # critère = F1 macro
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    report_to="none",
)

# Instanciation du Trainer pondéré avec early stopping (patience=2)
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    sample_weights=sample_weights,
)

In [ ]:
# Lancement de l'entraînement
trainer.train()

## 7. Matrice de confusion

On prédit sur l'ensemble de test et on affiche la matrice de confusion.

In [ ]:
# Prédictions sur le test set
predictions = trainer.predict(tokenized["test"])
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Matrice de confusion - Test set")
plt.xlabel("Prédiction"); plt.ylabel("Réel")
plt.tight_layout()
plt.show()

## 8. Analyse des erreurs

On affiche 20 exemples mal classifiés du test set avec leur texte, le label réel et le label prédit.

In [ ]:
# Récupération des textes originaux du test set (non tokenisés)
test_texts = dataset["test"]["text"]

# Indices des exemples mal classifiés
wrong_idx = np.where(y_pred != y_true)[0]

print(f"Nombre d'erreurs : {len(wrong_idx)} / {len(y_true)}\n")
errors = pd.DataFrame({
    "texte": [test_texts[i] for i in wrong_idx[:20]],
    "réel": [CLASS_NAMES[y_true[i]] for i in wrong_idx[:20]],
    "prédit": [CLASS_NAMES[y_pred[i]] for i in wrong_idx[:20]],
})
errors

## 9. Résultats finaux

Tableau récapitulatif : accuracy, F1 macro, F1 pondéré, et F1 par classe.

In [ ]:
# Métriques globales
acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

print(f"Accuracy     : {acc:.4f}")
print(f"F1 macro     : {f1_macro:.4f}")
print(f"F1 pondéré   : {f1_weighted:.4f}\n")

# Tableau du F1 par classe
f1_per = f1_score(y_true, y_pred, average=None, labels=list(range(NUM_LABELS)))
results = pd.DataFrame({"classe": CLASS_NAMES, "F1": np.round(f1_per, 4)})
print(results.to_string(index=False))

# Rapport complet (précision / rappel / F1)
print("\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

## 10. Test sur des phrases personnalisées

On teste le modèle sur 5 phrases en anglais pour vérifier qualitativement ses prédictions.

In [ ]:
# 5 phrases custom couvrant différentes émotions
custom_sentences = [
    "I can't stop smiling, today has been absolutely wonderful!",
    "I feel so alone and everything seems pointless lately.",
    "How dare you lie to me after everything I did for you!",
    "I'm terrified of what might happen during the surgery tomorrow.",
    "Wow, I never expected to win the lottery, this is unbelievable!",
]

# Tokenisation et prédiction
model.eval()
inputs = tokenizer(custom_sentences, truncation=True, padding=True,
                   max_length=128, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits
pred_labels = logits.argmax(dim=1).cpu().numpy()

# Affichage des prédictions
for sentence, label in zip(custom_sentences, pred_labels):
    print(f"[{CLASS_NAMES[label]:<8}] {sentence}")